In [1]:

import talib
import datetime as dt
import pandas as pd
import numpy as np
from vnstock import Vnstock
import pandas_ta as ta
import datetime as dt
import numpy as np
npNaN = np.nan

vnstock_instance = Vnstock().stock(symbol='ACB', source='TCBS')
stock = 'VNINDEX'
end_date = dt.datetime.now().strftime('%Y-%m-%d')
start_date = (dt.datetime.now() - dt.timedelta(days=7000)).strftime('%Y-%m-%d')

df = vnstock_instance.quote.history(symbol = stock ,start=start_date, end = end_date, interval='1D')
df.columns = df.columns.str.title()

# Hàm tính các chỉ báo
def trend_indicators(df):
    # 1. Bollinger Bands
    df['BB_upper'], df['BB_middle'], df['BB_lower'] = talib.BBANDS(df['Close'], timeperiod=20)
    # 2. candle sticks
    candle_names = talib.get_function_groups()['Pattern Recognition']
    for candle in candle_names:
        df[candle] = getattr(talib, candle)(df['Open'], df['High'], df['Low'], df['Close'])

    # 3. Simple Moving Average
    sma_periods = [10, 20, 50, 100, 200]
    for period in sma_periods:
        df[f'SMA_{period}'] = talib.SMA(df['Close'], timeperiod=period)

    
    # 4. Standard Deviation Channels (tự code vì không có sẵn trong talib)
    df['SMA'] = talib.SMA(df['Close'], timeperiod=20)
    df['STD'] = df['Close'].rolling(window=20).std()
    df['STD_upper'] = df['SMA'] + 2 * df['STD']
    df['STD_lower'] = df['SMA'] - 2 * df['STD']

    # 5. Trend Lines (tự code đơn giản - dùng hồi quy tuyến tính)
    def trend_line(data, window=20):
        x = np.arange(window)
        coeffs = np.polyfit(x, data[-window:], 1)
        return coeffs[0] * x + coeffs[1]
    df['Trend'] = df['Close'].rolling(window=20).apply(lambda x: trend_line(x)[-1], raw=False)

    # 6. Ultimate Oscillator
    df['UO'] = talib.ULTOSC(df['High'], df['Low'], df['Close'], timeperiod1=7, timeperiod2=14, timeperiod3=28)

    # 7. Weighted Moving Average
    df['WMA'] = talib.WMA(df['Close'], timeperiod=20)

    # 8. Wilder Moving Average (tự code vì không có sẵn trong talib)
    def wilder_ma(data, period=20):
        result = np.zeros(len(data))
        result[period-1] = np.mean(data[:period])
        for i in range(period, len(data)):
            result[i] = (result[i-1] * (period - 1) + data[i]) / period
        return result
    df['Wilder_MA'] = wilder_ma(df['Close'].values)

    # 9. Directional Movement (DMI)
    df['PLUS_DI'] = talib.PLUS_DI(df['High'], df['Low'], df['Close'], timeperiod=14)
    df['MINUS_DI'] = talib.MINUS_DI(df['High'], df['Low'], df['Close'], timeperiod=14)

    # 10. Displaced Moving Average (tự code - dịch chuyển SMA)
    df['DMA'] = talib.SMA(df['Close'], timeperiod=20).shift(5)

    # 11. Donchian Channels
    df['Donchian_upper'] = df['High'].rolling(window=20).max()
    df['Donchian_lower'] = df['Low'].rolling(window=20).min()

    # 12. Hull Moving Average (tự code)
    def hull_ma(data, period=9):
        wma1 = talib.WMA(data, timeperiod=period//2)  # Sửa: dùng 'timeperiod'
        wma2 = talib.WMA(data, timeperiod=period)  # Sửa: dùng 'timeperiod'
        raw_hma = 2 * wma1 - wma2
        hma = talib.WMA(raw_hma, timeperiod=int(np.sqrt(period)))  # Sửa: dùng 'timeperiod'
        return hma
    
    df['HMA'] = hull_ma(df['Close'].values, period=20)


    # 13. Ichimoku Cloud (dùng pandas_ta)
    ichimoku = df.ta.ichimoku()
    tenkan = 9
    kijun = 26
    senkou = 52
    shift = 26
    df['Tenkan-sen'] = (df['High'].rolling(window=tenkan).max() + df['Low'].rolling(window=tenkan).min()) / 2
    df['Kijun-sen'] = (df['High'].rolling(window=kijun).max() + df['Low'].rolling(window=kijun).min()) / 2
    df['Senkou Span A'] = ((df['Tenkan-sen'] + df['Kijun-sen']) / 2).shift(shift)
    df['Senkou Span B'] = ((df['High'].rolling(window=senkou).max() + df['Low'].rolling(window=senkou).min()) / 2).shift(shift)
    df['Chikou Span'] = df['Close'].shift(-shift)
    df['Chikou Span'] = df['Chikou Span'].fillna(0)


    # 15. Moving Average Oscillator (tự code: SMA ngắn - SMA dài)
    df['MA_Osc'] = talib.SMA(df['Close'], timeperiod=10) - talib.SMA(df['Close'], timeperiod=50)

    # 16. MACD Indicator
    df['MACD'], df['MACD_signal'], _ = talib.MACD(df['Close'], fastperiod=12, slowperiod=26, signalperiod=9)

    # 17. Moving Average (dùng EMA làm ví dụ)
    ema_periods = [10, 20, 50, 100, 200]
    for period in sma_periods:
        df[f'EMA_{period}'] = talib.EMA(df['Close'], timeperiod=period)
    # 18. Moving Average Filters (ví dụ: SMA làm filter)
    df['MA_Filter'] = np.where(df['Close'] > df['SMA'], 1, 0)

    # 19. Moving Average High/Low/Open
    df['MA_High'] = talib.SMA(df['High'], timeperiod=20)
    df['MA_Low'] = talib.SMA(df['Low'], timeperiod=20)
    df['MA_Open'] = talib.SMA(df['Open'], timeperiod=20)

    # 20. Rainbow 3D Moving Averages (tự code - nhiều SMA với độ dài khác nhau)
    for i in range(10, 31, 5):
        df[f'Rainbow_MA_{i}'] = talib.SMA(df['Close'], timeperiod=i)

    return df

def momentum_indicators(df):

    # 1. Accumulation Distribution
    df['AD'] = talib.AD(df['High'], df['Low'], df['Close'], df['Volume'])

    # 2. ADX
    df['ADX'] = talib.ADX(df['High'], df['Low'], df['Close'], timeperiod=14)

    # 3. Aroon Oscillator
    df['Aroon_Osc'] = talib.AROONOSC(df['High'], df['Low'], timeperiod=14)

    # 4. Bollinger %B (tự code: %B = (Close - BB_lower) / (BB_upper - BB_lower))
    upper, middle, lower = talib.BBANDS(df['Close'], timeperiod=20)
    df['BB_upper'] = upper
    df['BB_lower'] = lower
    df['Bollinger_%B'] = (df['Close'] - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'])

    # 5. Chaikin Money Flow (dùng pandas_ta)
    df['CMF'] = df.ta.cmf(length=20)

    # 6. Chaikin Oscillator (tự code: ADL ngắn - ADL dài)
    adl = talib.AD(df['High'], df['Low'], df['Close'], df['Volume'])
    df['Chaikin_Osc'] = talib.EMA(adl, timeperiod=3) - talib.EMA(adl, timeperiod=10)

    # 7. Relative Strength (Compare) (tự code: so sánh với benchmark, giả sử benchmark là SMA50)
    benchmark = talib.SMA(df['Close'], timeperiod=50)
    df['Relative_Strength'] = df['Close'] / benchmark

    # 8. Relative Strength Index (RSI)
    df['RSI'] = talib.RSI(df['Close'], timeperiod=14)

    # 9. Safezone Indicator (tự code - dựa trên hướng dẫn của Alexander Elder)
    def safezone(data, period=10):
        returns = data.pct_change()
        downside = np.minimum(returns, 0)
        return downside.rolling(window=period).mean() * -100
    df['Safezone'] = safezone(df['Close'])

    # 10. Stochastic Oscillator
    df['Stoch_K'], df['Stoch_D'] = talib.STOCH(df['High'], df['Low'], df['Close'], 
                                               fastk_period=14, slowk_period=3, slowd_period=3)

    # 11. Stochastic RSI
    df['Stoch_RSI'] = talib.STOCHRSI(df['Close'], timeperiod=14)[0]

    # 12. TRIX Indicator
    df['TRIX'] = talib.TRIX(df['Close'], timeperiod=15)

    # 13. Ultimate Oscillator
    df['UO'] = talib.ULTOSC(df['High'], df['Low'], df['Close'], timeperiod1=7, timeperiod2=14, timeperiod3=28)

    # 14. Volume Oscillator (tự code: Volume MA ngắn - Volume MA dài)
    df['Volume_Osc'] = talib.SMA(df['Volume'], timeperiod=5) - talib.SMA(df['Volume'], timeperiod=20)

    # 15. Williams %R
    df['Williams_%R'] = talib.WILLR(df['High'], df['Low'], df['Close'], timeperiod=14)

    # 16. Chande Momentum Oscillator
    df['CMO'] = talib.CMO(df['Close'], timeperiod=14)
     # 17. Compare Prices (tự code: so sánh Close với Open)
    df['Compare_Prices'] = df['Close'] - df['Open']

    # 18. Coppock Indicator (tự code: WMA của ROC ngắn + ROC dài)
    roc1 = df['Close'].pct_change(periods=11)
    roc2 = df['Close'].pct_change(periods=14)
    df['Coppock'] = talib.WMA(roc1 + roc2, timeperiod=10)

    # 19. Detrended Price Oscillator (tự code: Close - SMA dịch chuyển)
    sma = talib.SMA(df['Close'], timeperiod=20)
    df['DPO'] = df['Close'] - sma.shift(10)

    # 20. MACD Histogram
    macd, signal, hist = talib.MACD(df['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
    df['MACD'] = macd
    df['MACD_Signal'] = signal
    df['MACD_Hist'] = hist

    # 21. Momentum Indicator
    df['Momentum'] = talib.MOM(df['Close'], timeperiod=10)

    return df

def volume_indicators(df):
# 1. Accumulation Distribution
    df['AD'] = talib.AD(df['High'], df['Low'], df['Close'], df['Volume'])

    # 2. Chaikin Money Flow (dùng pandas_ta)
    df['CMF'] = df.ta.cmf(length=20)

    # 3. Chaikin Volatility (tự code: đo độ biến động của High-Low qua EMA)
    hl_range = df['High'] - df['Low']
    df['Chaikin_Volatility'] = talib.EMA(hl_range, timeperiod=10).pct_change(periods=10) * 100

    # 4. Rate of Change (Volume)
    df['ROC_Volume'] = talib.ROC(df['Volume'], timeperiod=10)

    # 5. Volume Oscillator (tự code: Volume MA ngắn - Volume MA dài)
    df['Volume_Osc'] = talib.SMA(df['Volume'], timeperiod=5) - talib.SMA(df['Volume'], timeperiod=20)

    # 6. Chandler Exits (tự code: dựa trên ATR và giá cao/thấp)
    atr = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)
    df['Chandler_Exit_Long'] = df['High'].rolling(window=10).max() - 3 * atr
    df['Chandler_Exit_Short'] = df['Low'].rolling(window=10).min() + 3 * atr

    # 7. Choppiness Index (tự code: đo mức độ "choppy" của thị trường)
    def choppiness_index(high, low, close, period=14):
        atr_sum = talib.ATR(high, low, close, timeperiod=1).rolling(window=period).sum()
        max_high = high.rolling(window=period).max()
        min_low = low.rolling(window=period).min()
        return 100 * np.log10(atr_sum / (max_high - min_low)) / np.log10(period)
    df['Choppiness'] = choppiness_index(df['High'], df['Low'], df['Close'])

    # 8. Commodity Channel Index (CCI)
    df['CCI'] = talib.CCI(df['High'], df['Low'], df['Close'], timeperiod=14)

    # 9. Negative Volume (tự code: Volume khi giá giảm)
    df['Negative_Volume'] = np.where(df['Close'] < df['Open'], df['Volume'], 0)

    # 10. Positive Volume (tự code: Volume khi giá tăng)
    df['Positive_Volume'] = np.where(df['Close'] > df['Open'], df['Volume'], 0)

    # 11. Rate of Change (Price)
    df['ROC_Price'] = talib.ROC(df['Close'], timeperiod=10)

    return df

def votatility_indicators(df):
    # 1. ATR Bands (tự code: SMA ± n * ATR)
    atr = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)
    sma = talib.SMA(df['Close'], timeperiod=20)
    df['ATR_Bands_Upper'] = sma + 2 * atr
    df['ATR_Bands_Lower'] = sma - 2 * atr

    # 2. ATR Trailing Stops (tự code: dựa trên ATR và hướng giá)
    def atr_trailing_stop(high, low, close, atr, multiplier=3):
        close = close.reset_index(drop=True)  # Reset index
        if close.isnull().any():
            raise ValueError("Chuỗi close có giá trị NaN, kiểm tra lại dữ liệu!")
        
        prev_close = close.shift(1)
        direction = np.where(close > prev_close, 1, -1)
        
        stop = np.full(len(close), np.nan)  # Khởi tạo với NaN
        stop[0] = close.iloc[0]  # Sửa lại cách truy cập phần tử đầu tiên
    
        for i in range(1, len(close)):
            if direction[i] == 1:
                stop[i] = max(stop[i-1], high.iloc[i] - multiplier * atr.iloc[i])
            else:
                stop[i] = min(stop[i-1], low.iloc[i] + multiplier * atr.iloc[i])
        
        return stop
    df['ATR_Trailing_Stop'] = atr_trailing_stop(df['High'], df['Low'], df['Close'], atr)

    # 3. Average True Range
    df['ATR'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)

    # 4. Bollinger Bands®
    df['BB_Upper'], df['BB_Middle'], df['BB_Lower'] = talib.BBANDS(df['Close'], timeperiod=20, nbdevup=2, nbdevdn=2)

    # 5. Bollinger Bands® Width (tự code: (Upper - Lower) / Middle)
    df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / df['BB_Middle']

    # 6. True Range
    df['True_Range'] = talib.TRANGE(df['High'], df['Low'], df['Close'])

    # 7. Vertical Horizontal Filter (VHF) (tự code: đo xu hướng vs sideways)
    def vhf(close, period=28):
        h_max = close.rolling(window=period).max()
        l_min = close.rolling(window=period).min()
        diff = h_max - l_min
        abs_change = close.diff().abs().rolling(window=period).sum()
        return diff / abs_change
    df['VHF'] = vhf(df['Close'])

    # 8. Volatility (tự code: độ lệch chuẩn của % thay đổi giá)
    df['Volatility'] = df['Close'].pct_change().rolling(window=20).std() * np.sqrt(252)  # Chuẩn hóa hàng năm

    # 9. Volatility Ratio (tự code: True Range / ATR)
    df['Volatility_Ratio'] = df['True_Range'] / df['ATR']

    # 10. Volatility Stops (tự code: dựa trên ATR và giá gần nhất)
    def volatility_stops(close, atr, multiplier=3):
        return close - multiplier * atr
    df['Volatility_Stops'] = volatility_stops(df['Close'], df['ATR'])

      # 11. Keltner Channels (dùng pandas_ta)
    keltner = df.ta.kc()
    df['Keltner_Upper'] = keltner.iloc[:, 1]  # Cột đầu tiên
    df['Keltner_Lower'] = keltner.iloc[:, 0]  # Cột thứ hai
    df['Keltner_Middle'] = (df['Keltner_Upper'] + df['Keltner_Lower']) / 2

    return df
    
def support_resist_indicators(df):
    # 1. Trend Lines (tự code: hồi quy tuyến tính đơn giản)
    def trend_line(data, window=20):
        x = np.arange(window)
        coeffs = np.polyfit(x, data[-window:], 1)
        return coeffs[0] * x[-1] + coeffs[1]
    df['Trend_Line'] = df['Close'].rolling(window=20).apply(lambda x: trend_line(x), raw=False)

    # 2. Donchian Channels
    df['Donchian_Upper'] = df['High'].rolling(window=20).max()
    df['Donchian_Lower'] = df['Low'].rolling(window=20).min()

    # 3. Fibonacci Extensions (tự code: dựa trên mức cao/thấp gần nhất)
    def fib_extensions(high, low, close):
        swing_high = high.rolling(window=20).max()
        swing_low = low.rolling(window=20).min()
        diff = swing_high - swing_low
        levels = [swing_high + diff * x for x in [0.618, 1.0, 1.618]]
        return pd.Series(levels[1], index=close.index)  # Ví dụ: mức 100%
    df['Fib_Extension'] = fib_extensions(df['High'], df['Low'], df['Close'])

    # 4. Fibonacci Retracements (tự code: các mức thoái lui)
    def fib_retracements(high, low):
        swing_high = high.rolling(window=20).max()
        swing_low = low.rolling(window=20).min()
        diff = swing_high - swing_low
        levels = [swing_high - diff * x for x in [0.236, 0.382, 0.5, 0.618]]
        return pd.Series(levels[2], index=high.index)  # Ví dụ: mức 50%
    df['Fib_Retracement'] = fib_retracements(df['High'], df['Low'])

    # 5. Heikin-Ashi Candlesticks (tự code)
    def heikin_ashi(df):
        ha_close = (df['Open'] + df['High'] + df['Low'] + df['Close']) / 4
        ha_open = pd.Series(np.nan, index=df.index)
        ha_open.iloc[0] = df['Open'].iloc[0]
        for i in range(1, len(df)):
            ha_open.iloc[i] = (ha_open.iloc[i-1] + ha_close.iloc[i-1]) / 2
        ha_high = pd.concat([df['High'], ha_open, ha_close], axis=1).max(axis=1)
        ha_low = pd.concat([df['Low'], ha_open, ha_close], axis=1).min(axis=1)
        return ha_open, ha_high, ha_low, ha_close
    df['HA_Open'], df['HA_High'], df['HA_Low'], df['HA_Close'] = heikin_ashi(df)

    # 6. Inverted Axis (tự code: đảo ngược giá - đơn giản hóa thành 1/Close)
    df['Inverted_Axis'] = 1 / df['Close']

    # 7. Exponential Moving Average
    df['EMA'] = talib.EMA(df['Close'], timeperiod=20)

    # 8. KST Indicator (tự code: tổng ROC với các trọng số)
    roc1 = df['Close'].pct_change(periods=10) * 1
    roc2 = df['Close'].pct_change(periods=15) * 2
    roc3 = df['Close'].pct_change(periods=20) * 3
    roc4 = df['Close'].pct_change(periods=30) * 4
    df['KST'] = talib.SMA(roc1 + roc2 + roc3 + roc4, timeperiod=9)

    # 9. Linear Regression Indicator (tự code: giá dự đoán từ hồi quy)
    def lin_reg(close, period=14):
        x = np.arange(period)
        return close.rolling(window=period).apply(lambda y: np.polyval(np.polyfit(x, y, 1), period-1), raw=False)
    df['Lin_Reg'] = lin_reg(df['Close'])

    # 10. Parabolic SAR
    df['PSAR'] = talib.SAR(df['High'], df['Low'], acceleration=0.02, maximum=0.2)

    # 11. Percentage Bands (tự code: SMA ± % giá)
    sma = talib.SMA(df['Close'], timeperiod=20)
    df['Percent_Bands_Upper'] = sma * 1.05  # +5%
    df['Percent_Bands_Lower'] = sma * 0.95  # -5%

    # 12. Percentage Trailing Stops (tự code: trailing stop dựa trên % giá)
    def percent_trailing_stop(close, percent=5):
        trailing = np.zeros(len(close))  # Mảng lưu giá trailing stop
        trailing[0] = close.iloc[0]  # Giá trị đầu tiên của trailing stop bằng giá mở cửa
    
        for i in range(1, len(close)):
            trailing[i] = max(trailing[i-1], close.iloc[i] * (1 - percent / 100))
    
        return pd.Series(trailing, index=close.index)  # Trả về Pandas Series có cùng index
    
    df['Percent_Trailing_Stop'] = percent_trailing_stop(df['Close'])

    # 13. Pivot Points (dùng pandas_ta)
    for window in [5, 20]:
        df[f'Pivot_{window}'] = df[['High', 'Low', 'Close']].mean(axis=1).rolling(window=window).mean()
        df[f'R1_{window}'] = (2 * df[f'Pivot_{window}']) - df['Low'].rolling(window=window).min()
        df[f'S1_{window}'] = (2 * df[f'Pivot_{window}']) - df['High'].rolling(window=window).max()
        df[f'R2_{window}'] = df[f'Pivot_{window}'] + (df['High'].rolling(window=window).max() - df['Low'].rolling(window=window).min())
        df[f'S2_{window}'] = df[f'Pivot_{window}'] - (df['High'].rolling(window=window).max() - df['Low'].rolling(window=window).min())

    # 14. Price Comparison (tự code: so sánh Close với SMA)
    df['Price_Comparison'] = df['Close'] - talib.SMA(df['Close'], timeperiod=20)

    # 15. Price Differential (tự code: Close - Open)
    df['Price_Diff'] = df['Close'] - df['Open']

    # 16. Price Envelope (tự code: SMA ± % cố định)
    df['Envelope_Upper'] = sma * 1.1  # +10%
    df['Envelope_Lower'] = sma * 0.9  # -10%

    # 17. Price Ratio (tự code: Close / SMA)
    sma_windows = [10, 20, 50, 100, 200]
    for window in sma_windows:
        df[f'Price_Ratio_{window}'] = df['Close'] / df['Close'].rolling(window=window).mean()

    return df

def another_indicator(df):
   # 1. Weighted Close (tự code: (High + Low + 2*Close) / 4)
    df['Weighted_Close'] = (df['High'] + df['Low'] + 2 * df['Close']) / 4

    # 2. Williams Accumulate Distribution (tự code: tích lũy dựa trên giá và volume)
    def williams_acc_dist(high, low, close, volume):
        trh = np.minimum(high, close.shift(1))
        trl = np.maximum(low, close.shift(1))
        ad = np.where(close > close.shift(1), close - trl,
                      np.where(close < close.shift(1), close - trh, 0))
        return ad * volume
    df['Williams_Acc_Dist'] = williams_acc_dist(df['High'], df['Low'], df['Close'], df['Volume']).cumsum()

    # 3. Williams Accumulation Distribution (tự code: phiên bản khác của Williams)
    def williams_accum_dist(high, low, close):
        buying = high - close.shift(1)
        selling = close.shift(1) - low
        return np.where(close > close.shift(1), buying,
                        np.where(close < close.shift(1), -selling, 0))
    df['Williams_Accum_Dist'] = williams_accum_dist(df['High'], df['Low'], df['Close']).cumsum()

    # 4. Ease of Movement (tự code: (High + Low)/2 - (High[-1] + Low[-1])/2) / (Volume / (High - Low)))
    def ease_of_movement(high, low, volume):
        distance = ((high + low) / 2 - (high.shift(1) + low.shift(1)) / 2)
        box_ratio = (volume / 1e8) / (high - low + 1e-10)  # Tránh chia cho 0
        return distance / box_ratio
    df['Ease_of_Movement'] = ease_of_movement(df['High'], df['Low'], df['Volume'])
        # Thay thế Inf bằng NaN để xử lý dễ hơn
    df['Ease_of_Movement'].replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Tìm vị trí các NaN (INF ban đầu)
    nan_indices = df[df['Ease_of_Movement'].isna()].index
    
    # Xử lý từng NaN theo quy tắc yêu cầu
    for i in range(len(nan_indices) - 1):
        idx = nan_indices[i]
        next_idx = nan_indices[i + 1]
        
        if next_idx - idx == 1:  # Hai INF liên tiếp
            df.loc[idx] = df.loc[idx - 1]  # INF 1 lấy từ bên trái
            df.loc[next_idx] = df.loc[next_idx + 1]  # INF 2 lấy từ bên phải
    
    # Xử lý INF lẻ bằng trung bình hai giá trị lân cận
    df['Ease_of_Movement'] = df['Ease_of_Movement'].interpolate(method='linear')


    # 5. Elder Ray Index (tự code: Bull Power và Bear Power)
    ema = talib.EMA(df['Close'], timeperiod=13)
    df['Bull_Power'] = df['High'] - ema
    df['Bear_Power'] = df['Low'] - ema

    # 6. Force Index
    df['Force_Index'] = talib.EMA((df['Close'] - df['Close'].shift(1)) * df['Volume'], timeperiod=13)

    # 7. MACD Percentage (tự code: (MACD / Close) * 100)
    macd, signal, _ = talib.MACD(df['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
    df['MACD_Percentage'] = (macd / df['Close']) * 100

    # 8. Mass Index (tự code: tổng EMA của High-Low range trong 25 ngày)
    hl_range = df['High'] - df['Low']
    ema1 = talib.EMA(hl_range, timeperiod=9)
    ema2 = talib.EMA(ema1, timeperiod=9)
    df['Mass_Index'] = ema1.rolling(window=25).sum() / ema2.rolling(window=25).sum()

    # 9. Median Price (tự code: (High + Low) / 2)
    df['Median_Price'] = (df['High'] + df['Low']) / 2

    # 10. Money Flow Index
    df['MFI'] = talib.MFI(df['High'], df['Low'], df['Close'], df['Volume'], timeperiod=14)

    # 11. On Balance Volume
    df['OBV'] = talib.OBV(df['Close'], df['Volume'])

    # 12. Price Compression (tự code: đo mức nén giá qua range nhỏ nhất trong n ngày)
    def price_compression(high, low, period=14):
        range_high = high.rolling(window=period).max()
        range_low = low.rolling(window=period).min()
        return (range_high - range_low) / range_low
    df['Price_Compression'] = price_compression(df['High'], df['Low'])

    return df
# Tính toán các chỉ báo
indicators_list = [
    trend_indicators(df),
    momentum_indicators(df),
    volume_indicators(df),
    votatility_indicators(df),
    support_resist_indicators(df),
    another_indicator(df)
]

indicators_df = pd.concat(indicators_list, axis=1)  # Gộp tất cả cột lại
indicators_df = indicators_df.T.drop_duplicates().T

# Drop 200 dòng NaN đầu tiên
indicators_df = indicators_df.iloc[200:]  
print(indicators_df.columns.tolist())

# Hiển thị kết quả
print(indicators_df)

Phiên bản Vnstock 3.2.3 đã có mặt, vui lòng cập nhật với câu lệnh : `pip install vnstock --upgrade`.
Lịch sử phiên bản: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban
Phiên bản hiện tại 3.2.1

2025-04-09 03:02:51 - vnstock.common.data.data_explorer - INFO - TCBS không cung cấp thông tin danh sách. Dữ liệu tự động trả về từ VCI.


DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, us

['Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'BB_upper', 'BB_middle', 'BB_lower', 'CDL2CROWS', 'CDL3BLACKCROWS', 'CDL3INSIDE', 'CDL3LINESTRIKE', 'CDL3OUTSIDE', 'CDL3STARSINSOUTH', 'CDL3WHITESOLDIERS', 'CDLABANDONEDBABY', 'CDLADVANCEBLOCK', 'CDLBELTHOLD', 'CDLBREAKAWAY', 'CDLCLOSINGMARUBOZU', 'CDLCONCEALBABYSWALL', 'CDLCOUNTERATTACK', 'CDLDARKCLOUDCOVER', 'CDLDOJI', 'CDLDOJISTAR', 'CDLDRAGONFLYDOJI', 'CDLENGULFING', 'CDLEVENINGDOJISTAR', 'CDLEVENINGSTAR', 'CDLGAPSIDESIDEWHITE', 'CDLGRAVESTONEDOJI', 'CDLHAMMER', 'CDLHANGINGMAN', 'CDLHARAMI', 'CDLHARAMICROSS', 'CDLHIGHWAVE', 'CDLHIKKAKE', 'CDLHIKKAKEMOD', 'CDLHOMINGPIGEON', 'CDLIDENTICAL3CROWS', 'CDLINNECK', 'CDLINVERTEDHAMMER', 'CDLKICKING', 'CDLKICKINGBYLENGTH', 'CDLLADDERBOTTOM', 'CDLLONGLEGGEDDOJI', 'CDLLONGLINE', 'CDLMARUBOZU', 'CDLMATCHINGLOW', 'CDLMORNINGDOJISTAR', 'CDLMORNINGSTAR', 'CDLONNECK', 'CDLPIERCING', 'CDLRICKSHAWMAN', 'CDLSEPARATINGLINES', 'CDLSHOOTINGSTAR', 'CDLSHORTLINE', 'CDLSPINNINGTOP', 'CDLSTALLEDPATTERN', 'CD

In [2]:
indicators_df = pd.concat(indicators_list, axis=1)
indicators_df = indicators_df.T.drop_duplicates().T
indicators_df = indicators_df.iloc[200:]
print("Kích thước indicators_df sau khi tính chỉ báo:", indicators_df.shape)
print(indicators_df.head())

Kích thước indicators_df sau khi tính chỉ báo: (4800, 191)
                    Time    Open    High     Low   Close   Volume    BB_upper  \
200  2006-11-22 00:00:00  610.16  610.16  610.16  610.16  3930900  600.743042   
201  2006-11-23 00:00:00  636.95  636.95  636.95  636.95  4670180  618.875038   
202  2006-11-24 00:00:00  665.53  665.53  665.53  665.53  6562950  641.572138   
203  2006-11-27 00:00:00  640.15  665.53  640.15  640.15  5094220  653.289317   
204  2006-11-28 00:00:00   638.0  640.15   638.0   638.0  4747480  662.542493   

    BB_middle    BB_lower CDL2CROWS  ... Ease_of_Movement Bull_Power  \
200  541.5035  482.263958         0  ...              0.0  46.392157   
201  547.1015  475.327962         0  ...              0.0  62.727563   
202   554.326  467.079862         0  ...              0.0  78.263625   
203  560.7135  468.137683         0  ...     -6322.306457  70.708822   
204  567.0365  471.530507         0  ...      -623.378087  39.160419   

    Bear_Power      F

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb
from bayes_opt import BayesianOptimization

# Giả sử indicators_df đã được tạo trước đó
indicators_df = indicators_df.rename(columns={'Time': 'Date'})
indicators_df["Date"] = pd.to_datetime(indicators_df["Date"])


# Hàm an toàn để tính % thay đổi giá
def safe_pct_change(series):
    # Thay thế 0 bằng giá trị nhỏ để tránh chia cho 0
    series = series.replace(0, 1e-10)
    # Tính % thay đổi và xử lý NaN/inf
    pct_change = series.pct_change().replace([np.inf, -np.inf], np.nan) * 100
    return pct_change

# 2. Tính % thay đổi giá ngày mai và gán nhãn
indicators_df['Price_Change_Pct'] = safe_pct_change(indicators_df['Close']).shift(-1)

def label_price_change(pct):
    if pd.isna(pct):  # Xử lý NaN
        return 1  # Gán nhãn "Keep" nếu không tính được
    if pct > 0:
        return 0  # Buy
    elif pct == 0:
        return 1  # Keep
    else:
        return 2  # Sell

indicators_df['Label'] = indicators_df['Price_Change_Pct'].apply(label_price_change)

# Danh sách features ban đầu
features = [
    'Date', 'Open', 'High', 'Low', 'Close', 'Volume',
    'BB_upper', 'BB_middle', 'BB_lower',
    'CDL2CROWS', 'CDL3BLACKCROWS', 'CDL3INSIDE', 'CDL3LINESTRIKE', 'CDL3OUTSIDE', 
    'CDL3STARSINSOUTH', 'CDL3WHITESOLDIERS', 'CDLABANDONEDBABY', 'CDLADVANCEBLOCK',
    'CDLBELTHOLD', 'CDLBREAKAWAY', 'CDLCLOSINGMARUBOZU',
    'CDLCOUNTERATTACK', 'CDLDARKCLOUDCOVER', 'CDLDOJI', 'CDLDOJISTAR', 'CDLDRAGONFLYDOJI',
    'CDLENGULFING', 'CDLEVENINGDOJISTAR', 'CDLEVENINGSTAR', 'CDLGAPSIDESIDEWHITE',
    'CDLGRAVESTONEDOJI', 'CDLHAMMER', 'CDLHANGINGMAN', 'CDLHARAMI', 'CDLHARAMICROSS',
    'CDLHIGHWAVE', 'CDLHIKKAKE', 'CDLHIKKAKEMOD', 'CDLHOMINGPIGEON', 'CDLIDENTICAL3CROWS',
    'CDLINNECK', 'CDLINVERTEDHAMMER', 'CDLKICKING', 'CDLKICKINGBYLENGTH', 'CDLLADDERBOTTOM',
    'CDLLONGLEGGEDDOJI', 'CDLLONGLINE', 'CDLMARUBOZU', 'CDLMATCHINGLOW', 'CDLMORNINGDOJISTAR',
    'CDLMORNINGSTAR', 'CDLONNECK', 'CDLPIERCING', 'CDLRICKSHAWMAN', 'CDLSEPARATINGLINES',
    'CDLSHOOTINGSTAR', 'CDLSHORTLINE', 'CDLSPINNINGTOP', 'CDLSTALLEDPATTERN', 'CDLSTICKSANDWICH',
    'CDLTAKURI', 'CDLTASUKIGAP', 'CDLTHRUSTING', 'CDLTRISTAR', 'CDLUNIQUE3RIVER',
    'CDLUPSIDEGAP2CROWS', 'CDLXSIDEGAP3METHODS',
    'SMA_10', 'SMA_50', 'SMA_100', 'SMA_200',
    'STD', 'STD_upper', 'STD_lower',
    'Trend', 'UO', 'WMA', 'Wilder_MA', 'PLUS_DI', 'MINUS_DI', 'DMA',
    'Donchian_upper', 'Donchian_lower', 'HMA', 'Tenkan-sen', 'Kijun-sen',
    'Senkou Span A', 'Senkou Span B', 'Chikou Span', 'MA_Osc',
    'MACD', 'MACD_signal', 'EMA_10', 'EMA_20', 'EMA_50', 'EMA_100', 'EMA_200',
    'MA_Filter', 'MA_High', 'MA_Low', 'MA_Open', 'Rainbow_MA_15', 'Rainbow_MA_25',
    'Rainbow_MA_30',
    'AD', 'ADX', 'Aroon_Osc', 'Bollinger_%B', 'CMF', 'Chaikin_Osc',
    'Relative_Strength', 'RSI', 'Safezone', 'Stoch_K', 'Stoch_D', 'Stoch_RSI',
    'TRIX', 'Volume_Osc', 'Williams_%R', 'CMO', 'Compare_Prices', 'Coppock', 'DPO',
    'MACD_Hist', 'Momentum', 'Chaikin_Volatility', 'ROC_Volume', 'Chandler_Exit_Long',
    'Chandler_Exit_Short', 'Choppiness', 'CCI', 'Negative_Volume', 'Positive_Volume',
    'ROC_Price', 'ATR_Bands_Upper', 'ATR_Bands_Lower', 'ATR_Trailing_Stop', 'ATR',
    'BB_Width', 'True_Range', 'VHF', 'Volatility', 'Volatility_Ratio', 'Volatility_Stops',
    'Keltner_Lower', 'Keltner_Middle', 'Fib_Extension', 'Fib_Retracement',
    'HA_Open', 'HA_High', 'HA_Low', 'HA_Close', 'Inverted_Axis', 'KST', 'Lin_Reg',
    'PSAR', 'Percent_Bands_Upper', 'Percent_Bands_Lower', 'Percent_Trailing_Stop',
    'Pivot_5', 'R1_5', 'S1_5', 'R2_5', 'S2_5',
    'Pivot_20', 'R1_20', 'S1_20', 'R2_20', 'S2_20',
    'Price_Comparison', 'Envelope_Upper', 'Envelope_Lower', 'Price_Ratio_10',
    'Price_Ratio_20', 'Price_Ratio_50', 'Price_Ratio_100', 'Price_Ratio_200',
    'Weighted_Close', 'Williams_Acc_Dist', 'Williams_Accum_Dist', 'Ease_of_Movement',
    'Bull_Power', 'Bear_Power', 'Force_Index', 'MACD_Percentage', 'Mass_Index',
    'Median_Price', 'MFI', 'OBV', 'Price_Compression'
]

# Thêm feature lag_1 và lag_2 cho tất cả feature trừ 'Date'
numeric_features = [f for f in features if f != 'Date']
for feature in numeric_features:
    indicators_df[f'{feature}_lag_1'] = indicators_df[feature].shift(1)
    indicators_df[f'{feature}_lag_2'] = indicators_df[feature].shift(2)

# Cập nhật danh sách features bao gồm cả các feature lag
features_extended = features + [f'{feat}_lag_1' for feat in numeric_features] + [f'{feat}_lag_2' for feat in numeric_features]

# Chuyển đổi tất cả các cột kiểu object sang float nếu có thể, trừ cột Date
indicators_df[features_extended] = indicators_df[features_extended].apply(pd.to_numeric, errors='coerce')

# Loại bỏ các hàng có NaN do shift tạo ra

X = indicators_df[features_extended]
y = indicators_df['Label']

# 6. Chia dữ liệu train/test
split_index = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# 7. Chuyển sang DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# 1. Định nghĩa hàm mục tiêu cho Bayesian Optimization dựa trên accuracy
# 1. Định nghĩa hàm mục tiêu cho Bayesian Optimization dựa trên accuracy
# 1. Định nghĩa hàm mục tiêu cho Bayesian Optimization dựa trên merror
def xgb_evaluate(max_depth, learning_rate, subsample, colsample_bytree, alpha, lambda_):
    params = {
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'merror',
        'max_depth': int(max_depth),
        'eta': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'alpha': alpha,          # L1 regularization
        'lambda': lambda_,       # L2 regularization (changed from lambda_ to lambda)
        'seed': 42
    }
    cv_results = xgb.cv(params, dtrain, num_boost_round=100, nfold=3, metrics='merror', early_stopping_rounds=10)
    accuracy = 1 - cv_results['test-merror-mean'].iloc[-1]
    return accuracy

# 2. Thiết lập không gian tìm kiếm, bao gồm alpha và lambda
pbounds = {
    'max_depth': (3, 10),
    'learning_rate': (0.01, 0.3),
    'subsample': (0.5, 1),
    'colsample_bytree': (0.5, 1),
    'alpha': (0, 1),         # Phạm vi cho L1
    'lambda_': (0, 2)        # Phạm vi cho L2
}

# 3. Chạy Bayesian Optimization
optimizer = BayesianOptimization(f=xgb_evaluate, pbounds=pbounds, random_state=42)
optimizer.maximize(init_points=5, n_iter=20)

# 4. Lấy tham số tối ưu
best_params = optimizer.max['params']
best_params['max_depth'] = int(best_params['max_depth'])
best_params['objective'] = 'multi:softprob'
best_params['num_class'] = 3
best_params['eval_metric'] = 'merror'
best_params['seed'] = 42
best_params['alpha'] = best_params['alpha']
best_params['lambda'] = best_params['lambda_']  # Changed from lambda_ to lambda

evals = [(dtrain, 'train'), (dtest, 'test')]
# 5. Huấn luyện mô hình với tham số tối ưu
bst = xgb.train(best_params, dtrain, num_boost_round=100, evals=evals, early_stopping_rounds=10, verbose_eval=10)

# Dự đoán và tính accuracy
y_pred_prob = bst.predict(dtest)  # Xác suất cho từng lớp
y_pred = np.argmax(y_pred_prob, axis=1)  # Nhãn dự đoán
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on test set: {accuracy:.4f}")

# Tạo DataFrame kết quả
df_result = X_test.copy()
df_result['Close'] = indicators_df.loc[X_test.index, 'Close']
df_result['Date'] = indicators_df.loc[X_test.index, 'Date']
df_result['y_true'] = y_test.values
df_result['y_pred'] = y_pred
df_result['Signal'] = df_result['y_pred'].map({0: 'Buy', 1: 'Keep', 2: 'Sell'})

# Hiển thị kết quả
display_cols = ['Date', 'Close', 'y_true', 'y_pred', 'Signal']
df_display = df_result.reset_index()[display_cols]
print(df_display)

# Dự đoán cho ngày mai
last_row = indicators_df[features_extended].iloc[-1:].copy()
dlast = xgb.DMatrix(last_row)
y_pred_prob_last = bst.predict(dlast)  # Xác suất cho ngày mai
y_pred_last = np.argmax(y_pred_prob_last, axis=1)[0]  # Nhãn dự đoán

df_prediction = pd.DataFrame({
    'Date': [indicators_df['Date'].iloc[-1]],
    'Close': [indicators_df['Close'].iloc[-1]],
    'y_pred': [y_pred_last],
    'Signal': [{0: 'Buy', 1: 'Keep', 2: 'Sell'}[y_pred_last]],
    'y_true': [None]
})
print("\nDự đoán cho ngày mai:")
print(df_prediction)

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newfr

|   iter    |  target   |   alpha   | colsam... |  lambda_  | learni... | max_depth | subsample |
-------------------------------------------------------------------------------------------------
| 1         | 0.5674    | 0.3745    | 0.9754    | 1.464     | 0.1836    | 4.092     | 0.578     |
| 2         | 0.5565    | 0.05808   | 0.9331    | 1.202     | 0.2153    | 3.144     | 0.985     |
| 3         | 0.5654    | 0.8324    | 0.6062    | 0.3636    | 0.06319   | 5.13      | 0.7624    |
| 4         | 0.5664    | 0.4319    | 0.6456    | 1.224     | 0.05045   | 5.045     | 0.6832    |
| 5         | 0.5549    | 0.4561    | 0.8926    | 0.3993    | 0.1591    | 7.147     | 0.5232    |
| 6         | 0.5617    | 0.9864    | 0.8661    | 1.554     | 0.263     | 4.509     | 0.5419    |
| 7         | 0.5659    | 0.3646    | 0.6518    | 1.26      | 0.03822   | 5.03      | 0.7491    |
| 8         | 0.5721    | 0.168     | 0.8702    | 0.9151    | 0.0271    | 4.566     | 0.5332    |
| 9         | 0.5643

[03:10:27] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "lambda_" } are not used.



[0]	train-merror:0.40052	test-merror:0.45312
[10]	train-merror:0.35391	test-merror:0.46042
[17]	train-merror:0.34922	test-merror:0.46042
Accuracy on test set: 0.5396
                    Date    Close  y_true  y_pred Signal
0    1648512000000000000  1497.76       2       0    Buy
1    1648598400000000000  1490.51       0       0    Buy
2    1648684800000000000  1492.15       0       0    Buy
3    1648771200000000000  1516.44       0       0    Buy
4    1649030400000000000  1524.70       2       0    Buy
..                   ...      ...     ...     ...    ...
955  1743465600000000000  1317.33       0       2   Sell
956  1743552000000000000  1317.83       2       2   Sell
957  1743638400000000000  1229.84       2       2   Sell
958  1743724800000000000  1210.67       2       0    Buy
959  1744070400000000000  1132.79       1       0    Buy

[960 rows x 5 columns]

Dự đoán cho ngày mai:
                  Date    Close  y_pred Signal y_true
0  1744070400000000000  1132.79       0    Buy   